<div align="center">

# 🇳🇵 Nepal GovAgent — Live Demo
### Ask Nepal's government documents anything. In Nepali or English.

[![PyPI](https://img.shields.io/pypi/v/nepal-gov-agent?color=teal)](https://pypi.org/project/nepal-gov-agent/)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)
[![GitHub](https://img.shields.io/badge/GitHub-irfanalidv%2FNepal--Gov--Agent-black?logo=github)](https://github.com/irfanalidv/Nepal-Gov-Agent)

**Built by [Irfan Ali](https://github.com/irfanalidv) · DataCortex IQ**

</div>

---

## 📁 What's available — 5 documents via `download_corpus()`

The next step downloads five Nepal government PDFs into `./nepal_gov_data/` (no separate repo checkout). Example questions you can try:

| Document | Language | Example questions |
|---|---|---|
| **National AI Policy 2082** | English | Data privacy · AI governance |
| **Constitution of Nepal** (2nd amendment) | English | Fundamental rights |
| **Digital Nepal Framework 2.0** | English | Digitization priorities |
| **प्रतिनिधि सभा निर्वाचन अध्यादेश २०८२** | Nepali 🇳🇵 | Candidate eligibility · voting |
| **मानव अधिकार पुरस्कार कोष नियमावली** | Nepali 🇳🇵 | Fund purpose · awards |

> 💡 **Add your own PDFs?** See **Use Your Own Nepal Government PDFs** near the end.

---

## How it works

Every answer includes **source citations** — document name and page. Runs offline on this machine; nothing is sent to the cloud.

> ⏱️ **First run:** Install + download takes ~2 minutes. After that, answers are instant.



> ⚠️ **Research Prototype** — This is not production-ready for government deployment. The benchmark measures retrieval quality only, not answer safety or factual correctness. See [Scope and Limitations](https://github.com/irfanalidv/Nepal-Gov-Agent#scope-and-limitations) in the README before any official use.



## 🛠️ Setup — Run once


In [1]:
# Colab-friendly install from GitHub main (includes unreleased allowed_doc_ids)
import subprocess, sys, os
os.environ['PIP_DISABLE_PIP_VERSION_CHECK'] = '1'
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '-q', 'protobuf>=4,<5'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '-q', 'git+https://github.com/irfanalidv/Nepal-Gov-Agent.git'])

print("✅ Installation complete")


✅ Installation complete


In [2]:
# ------------------------------------------------------------
# Option A — Download the seed corpus (same PDFs as repo Data/)
# 5 documents: AI Policy, Constitution, Digital Nepal Framework,
# election ordinance (Nepali), human rights fund rules (Nepali).
# (Seed omits the Law Commission Legal Maxims volume — see README.)
# ------------------------------------------------------------
import warnings, os, logging

warnings.filterwarnings("ignore")
os.environ["USER_AGENT"] = "nepal-gov-agent-demo/1.0"
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)

from nepal_gov_agent import GovRAG, GovRAGConfig, download_corpus

corpus_dir = download_corpus()  # downloads to ./nepal_gov_data/

# Older seed downloads included a large Legal Maxims PDF — remove if present
_legacy_maxims = os.path.join(corpus_dir, "1714977234_32.pdf")
if os.path.isfile(_legacy_maxims):
    os.remove(_legacy_maxims)
    print("Removed legacy Legal Maxims PDF from corpus folder")

# ------------------------------------------------------------
# Option B — Use your own Nepal government PDFs
# Comment out Option A above, uncomment below, point at your folder
# ------------------------------------------------------------
# corpus_dir = "./my_ministry_docs/"

_cfg = GovRAGConfig(cache_dir=os.path.join(corpus_dir, ".nepal_gov_cache"))
rag = GovRAG(corpus_dir=corpus_dir, config=_cfg)
print(f"✅ GovRAG ready — corpus: {corpus_dir}")
print("🔒 Fully offline. No data leaves this machine.")


📁 Corpus folder: ./nepal_gov_data
📥 Downloading 5 Nepal government documents...

   ✓ Already exists — skipping: 2082.9.2 प्रतिनिधि सभा सदस्य निर्वाचन (पहिलो संशोधन) अध्यादेश,२०८२_v1cs5ms.pdf
   ✓ Already exists — skipping: Constitution of Nepal (2nd amd. English)_xf33zb3.pdf
   ✓ Already exists — skipping: National AI Policy-Final_uxc94vg.pdf
   ✓ Already exists — skipping: dnf_jbji8eb.pdf
   ✓ Already exists — skipping: मानव अधिकार पुरस्कार कोष सञ्चालन नियमावली, 2075_n4hme7v.pdf

✅ Seed corpus ready — 5 documents in ./nepal_gov_data
💡 To use your own PDFs, pass any folder to GovRAG(corpus_dir=...)



✅ GovRAG ready — corpus: ./nepal_gov_data
🔒 Fully offline. No data leaves this machine.


## 🔍 Live Demos — Ask Nepal's Government Documents
*Each demo below shows a real business scenario. Run any cell to see live retrieval.*

> **Optional — answer synthesis:** Install [Ollama](https://ollama.com), run `ollama pull qwen2.5:7b`, then set `use_llm=True` in `ask_and_display(...)` for concise answers with inline citations.



In [3]:
# Helper — run this once before the demo cells below
from IPython.display import Markdown, display
import re

_DOC_ALIASES = {
    "dnf_jbji8eb.pdf": "Digital Nepal Framework 2.0",
}


def _clean_doc_display(doc_id: str) -> str:
    if doc_id in _DOC_ALIASES:
        return _DOC_ALIASES[doc_id]
    base = doc_id.replace(".pdf", "").replace(".PDF", "")
    if "_" in base:
        prefix, suffix = base.rsplit("_", 1)
        if suffix.isalnum() and len(suffix) >= 5:
            return prefix
    return base


def ask_and_display(question, context="", *, rag_client=None, use_llm=False, allowed_doc_ids=None):
    """If use_llm=True, tries Ollama (qwen2.5:7b) for grounded synthesis with citations."""
    g = rag if rag_client is None else rag_client
    if context:
        display(Markdown(f"> 💼 **Scenario:** {context}"))
    display(Markdown(f"**❓ Question:** `{question}`"))

    if use_llm:
        try:
            from nepal_gov_agent import OllamaClient

            result = g.ask(
                question,
                llm=OllamaClient(),
                with_citations=True,
                allowed_doc_ids=allowed_doc_ids,
            )
        except Exception as e:
            display(Markdown(f"*LLM unavailable ({e!s}) — showing retrieval-only answer.*"))
            result = g.ask(question, allowed_doc_ids=allowed_doc_ids)
    else:
        result = g.ask(question, allowed_doc_ids=allowed_doc_ids)

    answer = result.answer or ""
    if not use_llm:
        answer = re.sub(r"\[[^\]]*?\]\s*Block\s*\d+\s*\n", "", answer)
        answer = re.sub(
            r"^www\.lawcommission\.gov\.np\s*\n", "", answer, flags=re.MULTILINE
        )
        answer = re.sub(r"^\d+\s*\n", "", answer, flags=re.MULTILINE)
        answer = re.sub(r"\n{3,}", "\n\n", answer).strip()
        if len(answer) > 500:
            truncated = answer[:500]
            last_period = truncated.rfind(".")
            if last_period > 200:
                answer = truncated[: last_period + 1] + "\n\n*[See source documents for full text]*"
            else:
                answer = truncated + "…\n\n*[See source documents for full text]*"
    else:
        answer = re.sub(r"\[[^\]]*?\]\s*Block\s*\d+\s*\n", "", answer)
        answer = re.sub(
            r"^www\.lawcommission\.gov\.np\s*\n", "", answer, flags=re.MULTILINE
        )
        answer = re.sub(r"^\d+\s*\n", "", answer, flags=re.MULTILINE)
        answer = re.sub(r"\n{3,}", "\n\n", answer).strip()

        # When the model falls back to retrieval-style text, jump to the first
        # query-matching keyword so previews show the most relevant section first.
        tokens = [
            t.lower()
            for t in re.findall(r"[A-Za-z]{4,}", question)
            if t.lower() not in {"what", "does", "about", "the", "and", "with", "from", "that"}
        ]
        first = -1
        low = answer.lower()
        for t in tokens:
            i = low.find(t)
            if i != -1 and (first == -1 or i < first):
                first = i
        if first > 180:
            start = answer.rfind("\n", 0, first)
            answer = answer[start + 1 :].strip() if start != -1 else answer[first:].strip()

    display(Markdown("---"))
    display(Markdown("**📋 Answer:**"))
    display(Markdown(answer))
    display(Markdown("---"))
    display(Markdown("**📌 Sources:**"))
    for src in result.sources[:3]:
        doc_clean = _clean_doc_display(src["doc"])
        display(Markdown(f"- 📄 `{doc_clean}` — Page {src['page']}"))
    display(Markdown(""))
    return result


# Optional: cleaner answers with a local LLM (fully offline on your machine):
#   brew install ollama  # or see https://ollama.com
#   ollama pull qwen2.5:7b
# Then call: ask_and_display("...", use_llm=True)



---

### Usecase 1 — National AI Policy (English)

**Scenario:** Tech entrepreneur checking data governance before launching in Nepal.

**Question:** *What does Nepal's National AI Policy say about data privacy and sovereignty?*



In [4]:
_ = ask_and_display(
    "What does Nepal's National AI Policy say about AI governance framework and data classification?",
    context="Tech entrepreneur checking data governance before launching in Nepal.",
    use_llm=True,
    allowed_doc_ids={"pdf:National AI Policy-Final_uxc94vg.pdf"},
)


> 💼 **Scenario:** Tech entrepreneur checking data governance before launching in Nepal.

**❓ Question:** `What does Nepal's National AI Policy say about AI governance framework and data classification?`

---

**📋 Answer:**

9.4  Develop standards for data, algorithms, and technologies, ensuring 
alignment with ethical values in the development and use of AI. 
9.5  Undertake benchmarking, standardization, and certification of AI systems 
developed and used in Nepal to ensure quality, safety, and reliability, and 
develop a National AI Index. 
9.6  Develop standards to manage challenges such as intellectual property 
misuse, cybersecurity risks, and the spread of misinformation or 
disinformation through AI. 
Related to Strategy 8.2 (Establish specialized structures for the operation, 
regulation, and promotion of AI) 
9.7  Establish an AI Regulatory Council to provide guidance on AI research, 
development, and application in line with international principles and 
practices. 
9.8  Establish a National AI Centre to ensure effective implementation of AI-
related policies and laws. 
9.9  Set up AI Excellence Centres in universities, research institutions, and 
educational institutions for AI studies, research, and development. 
9.10 Create a Regulatory Sandbox for the safe development and testing of AI 
systems. 
9.11 Enhance the capacity of existing institutions to address challenges such as 
privacy, ethics, human rights, and cybersecurity risks arising from use of 
AI.

---

driven farming system, crop and disease monitoring, smart irrigation and 
pesticide use, livestock health monitoring, and e-markets to modernize and 
commercialize agriculture. 
9.30 Reduce learning inequality through the application of natural language 
processing (NLP), personalized learning, and adaptive learning. 
9.31 Enhance the accessibility and quality of healthcare services through AI 
applications in areas like diagnostic imaging, early disease detection, and 
genomic analysis. 
9.32 Increase the use of AI-based technologies such as smart grid, smart 
switching, and smart meter in energy production, transmission, and 
distribution. 
9.33 Improve road safety and reduce cost and time through application of AI in 
areas like traffic management, parking management, traffic surveillance, 
public transport, and logistics. 
9.34 Promote Nepal’s tourism through AI-based destination selection, virtual 
tour guide, 3D modeling and enhanced tourist safety. 
9.35 Enhance the efficiency, productivity and effectiveness of the financial 
sector through the use of AI in areas like promotion of institutional 
governance of banks and financial institutions, control of financial crimes 
and prevention of money laundering. 
9.36 Improve production, productivity, and quality in the industrial sector 
through application of modern technologies like AI automation, automated 
system and Industry 4.0. 
9.37 Maximize usage of AI in environmental and natural resource conservation, 
hydrological and meteorological forecasting, management of disasters like 
earthquakes, floods, landslides, wildfires etc. and control of pollution.

---

9.49 Use AI to enhance easier access to public service delivery for women, 
children, senior citizens, marginalized communities, minorities, and 
persons with disabilities. 
9.50 Expand AI use in citizen security, crime investigation and surveillance, and 
emergency response. 
9.51 Maximize AI use for the effective implementation of the Digital Nepal 
Framework. 
9.52 Apply AI to enhance the effectiveness of services delivered through the 
Nagarik App. 
Related to Strategy 8.10 (Develop a risk management and AI governance 
framework in line with globally recognized standards) 
9.53 Prepare and implement an AI governance framework to classify AI systems 
based on risk and mitigate risks accordingly. 
9.54 Ensure classification of data used in AI, sectoral data collection, dataset 
creation and make provision of secure and easy access to data storage. 
9.55 Promote open data exchange and interoperability. 
9.56 Establish an AI-focused open data portal to provide quality datasets in 
sectors such as agriculture, education, health, industry, and tourism, thereby 
promoting development and innovation. 
9.57 Maintain access and control of relevant authorities over sensitive data 
required for AI research and development. 
9.58 Ensure necessary measures for data security and privacy in AI systems. 
9.59 Coordinate and collaborate with AI stakeholders to protect trademarks, 
patents, and intellectual property during AI usage/application. 
9.60 Adopt preventive and control measures to mitigate negative impacts caused 
by misinformation, deepfakes, and personal biases through AI.

---

The AI Regulation Council shall meet at least twice a year. The AI Regulation 
Council may invite officials or employees in its meeting as required. The tenure of 
members under Section 10.1 (h) shall be three years from the date of appointment. 
The National AI Centre shall serve as the Secretariat of the AI Regulation Council. 
10.1.1 Functions, Duties and Powers of the AI Regulation Council 
(a)  
Provide guidance focusing on fairness, transparency, accountability, 
protection of intellectual property, and protection of human rights in AI 
development and use, and approve related standards. 
(b)  
Recommend to the Government of Nepal to align relevant national laws 
and standards with international treaties and norms. 
(c)  
Facilitate necessary coordination and collaboration among federal, 
provincial and local levels, and other relevant agencies on AI 
development and use. 
(d)  
Test and ensure compliance with approved AI policies, laws, and 
standards. 
(e)  
Provide necessary guidance for effective policy implementation. 
(f)  
Perform other necessary tasks. 
10.2 National AI Centre 
A National AI Centre shall be established under the Ministry of Communication 
and Information Technology to promote, encourage, facilitate, and regulate the 
study, research, development, and application of AI. 
10.2.1 Functions, Duties and Powers of the National AI Centre 
(a)  
Promote, encourage, and regulate the study, research, development, and 
application of AI. 
(b)  
Act as the focal point for coordination and cooperation on AI-related 
matters at the international level.

---

2.2 Need for Policy 
The need for this policy arises from the necessity of internalizing the fundamental 
principles of human-Centered and ethical AI, as guided by the United Nations 
Educational, Scientific and Cultural Organization (UNESCO) and fostering the 
development and use of ethical, responsible, and transparent AI systems. Also, this 
policy is required to coordinate with national and international organizations 
related to AI; to produce skilled human resources in information technology; to 
create new employment opportunities through AI-based industries and startup; to 
prioritize AI-related research and development, and its application across diverse 
sectors; and the promotion of innovation to generate new opportunities. This policy 
is also necessary to address risks associated with AI, such as the spread of 
misinformation and deepfakes, job displacement and its social and economic 
consequences, and to establish mechanisms for regulation, identification and 
categorization of risks that can arise from the use of this technology, and its 
mitigation.  
Furthermore, this policy is necessary to promote good governance by integrating 
AI into existing information technology systems in Nepal, to enhance the quality, 
accessibility, and efficiency of public service delivery across all three tiers of 
government in sectors such as agriculture, education, health, industry, finance, 
public services, and security. It is also required to ensure the creation, availability, 
and accessibility of data while maintaining privacy and security, to develop policy 
and structural mechanisms for building a robust data ecosystem, and ultimately to 
achieve comprehensive socio-economic development through the effective 
utilization of Artificial Intelligence.

---

**📌 Sources:**

- 📄 `National AI Policy-Final` — Page 10

- 📄 `National AI Policy-Final` — Page 13

- 📄 `National AI Policy-Final` — Page 15

---

### Usecase 2 — Constitutional rights (English)

**Scenario:** NGO designing a citizen services program.

**Question:** *What fundamental rights does the Constitution of Nepal guarantee?*



In [5]:
_ = ask_and_display(
    "What fundamental rights do citizens have in Part 3 of the Constitution?",
    context="NGO designing a citizen services program.",
    use_llm=True,
    allowed_doc_ids={"pdf:Constitution of Nepal (2nd amd. English)_xf33zb3.pdf"},
)


> 💼 **Scenario:** NGO designing a citizen services program.

**❓ Question:** `What fundamental rights do citizens have in Part 3 of the Constitution?`

---

**📋 Answer:**

Right relating to Education: (1) Every citizen shall have the right of access to basic 
education. 
 
 
(2) Every citizen shall have the right to get compulsory and free education up 
to the basic level and free education up to the secondary level from the State. 
 
 
(3) The citizens with disabilities and the indigent citizens shall have the right 
to get free higher education in accordance with law. 
 
 
(4) The visually impaired citizens shall have the right to get free education 
through brail script and the citizens with hearing or speaking impairment, to get free 
education through sign language, in accordance with law. 
 (5) Every Nepali community residing in Nepal shall have the right to get 
education in its own mother tongue and, for that purpose, to open and operate schools 
and educational institutes, in accordance with law. 
32. 
Right to Language and Culture: (1) Every person and community shall have the right 
to use their own languages. 
 
 
(2) Every person and community shall have the right to participate in the 
cultural life of his/her/its community. 
(3) Every Nepali community residing in Nepal shall have the right to preserve 
and promote its language, script, culture, cultural civilization and heritage. 
33. 
Right to Employment: (1) Every citizen shall have the right to employment. The terms 
and conditions of employment, and unemployment benefit shall be as provided for in 
the federal law. 
(2) Every citizen shall have the right to choose employment. 
34. 
Right to Labour: (1) Every labourer shall have the right to fair labour practice.  
Explanation: For the purposes of this Article, "labourer" means a labourer or 
worker who does physical or intellectual work for an employer in consideration for 
remuneration.

---

14. 
Distribution of house and land ownership certificates 
15. 
Agriculture and animal husbandry, agro-products management, animal health, 
cooperatives 
16. 
Management of senior citizens, persons with disabilities and the incapacitated 
17. 
Collection of statistics of the unemployed 
18. 
Management, operation and control of agricultural extension 
19. 
Water supply, small hydropower projects, alternative energy 
20. 
Disaster management 
21. 
Conservation of watersheds, wildlife, mines and minerals 
22. 
Protection and development of languages, cultures and fine arts

---

(a) 
To safeguard the nationality, sovereignty and integrity of Nepal, while 
being loyal to the nation; 
(b) 
To abide by the Constitution and law; 
(c) 
To render compulsory service as and when the State so requires; 
(d) 
To protect and preserve public property.

---

(3) 
To ensure reliable supply of energy in an affordable, and easy manner, 
and make proper use of energy, for the fulfilment of the basic needs of 
citizens, by generating and developing renewable energy; 
(4) 
To develop sustainable and reliable irrigation by making control of 
water-induced disasters, and river management; 
(5) 
To conserve, promote, and make sustainable use of, forests, wildlife, 
birds, vegetation and bio-diversity, by mitigating possible risks to 
environment from industrial and physical development, while raising 
awareness of public-in-general about environment cleanliness; 
(6) 
To maintain forest area in territory required for ecological balance; 
(7) 
To adopt appropriate measures to abolish or mitigate existing or 
possible adverse environmental impacts on the nature, environment or 
biological diversity; 
(8) 
To pursue principles of environmentally sustainable development such 
as the principles of polluter pay, of precaution in environmental 
protection and of prior informed consent; 
(9) 
To make provisions of advance warning, preparedness, rescue, relief 
and rehabilitation in order to mitigate risks from natural disasters. 
(h) 
Policies relating to Basic Needs of Citizens: 
(1) 
To prepare human resources that are competent, competitive, ethical, 
and devoted to national interests, while making education scientific, 
technical, vocational, skill-oriented, employment and people-oriented; 
(2) 
To promote investment of the State in education sector and to make 
private sector investment in education service-oriented by regulation 
and management; 
 (3) 
To make higher education easy, qualitative and accessible, and free 
gradually; 
(4) 
To establish and promote community information centres and libraries 
for personality development of citizens;

---

(13) 
To make planned investment in sports and sport-persons in order to 
prepare healthy, competent and disciplined citizens, and to develop 
sports as a means of consolidating national unity and enhancing 
national prestige at the international level; 
(14) 
To adopt a single door system for the establishment, approval, operation, 
regulation and management of community-based and national or 
international non-governmental organizations and to involve such 
organizations only in the sectors of national need and priority, while 
making investment and role of such organizations transparent and 
accountable. 
(k) 
Policies relating to Justice and Penal System: 
(1) 
To make administration of justice speedy, efficient, widely available, 
cost-effective, impartial, effective, and accountable to people; 
(2) 
To pursue alternative means such as mediation and arbitration for the 
settlement of disputes of ordinary nature; 
(3) 
To adopt effective measures for control of corruption and irregularities 
in all sectors including political, administrative, judicial and social 
sectors. 
(l) 
Policies relating to Tourism: To develop eco-friendly tourism industries as an 
important base of national economy by way of identification, protection, 
promotion and publicity of the   historical, cultural, religious, archaeological 
and natural heritages of Nepal, to make environment and policy required for 
the development of tourism culture, and to accord priority to local people in 
the distribution of benefits of tourism industries.  
(m) 
Policies relating to International Relations: 
(1) 
To conduct independent  foreign policy based on the Charter of the 
United Nations, non-alignment, principles of Panchsheel, international 
law and the norms of world peace, taking into consideration of the 
overall interest of  the nation, while remaining active in safeguarding 
the sovereignty, territorial integrity, independence and national interest 
of Nepal;

---

**📌 Sources:**

- 📄 `Constitution of Nepal (2nd amd. English)` — Page 16

- 📄 `Constitution of Nepal (2nd amd. English)` — Page 201

- 📄 `Constitution of Nepal (2nd amd. English)` — Page 21

---

### Usecase 3 — Election ordinance (Nepali 🇳🇵)

**Scenario:** Municipality officer in Butwal checking candidate eligibility rules.

**Question:** *प्रतिनिधि सभाको निर्वाचनमा उम्मेदवार हुन के-के योग्यता चाहिन्छ?*

*(What qualifications are needed to be a candidate in the House of Representatives election?)*



In [6]:
_ = ask_and_display(
    "निर्वाचनमा उम्मेदवार हुन के चाहिन्छ?",
    context="Municipality officer in Butwal checking candidate eligibility rules.",
    use_llm=True,
    allowed_doc_ids={"pdf:2082.9.2 प्रतिनिधि सभा सदस्य निर्वाचन (पहिलो संशोधन) अध्यादेश,२०८२_v1cs5ms.pdf"},
)


> 💼 **Scenario:** Municipality officer in Butwal checking candidate eligibility rules.

**❓ Question:** `निर्वाचनमा उम्मेदवार हुन के चाहिन्छ?`

---

**📋 Answer:**

प्रतितिति सभा सदस्य तिर्ाचि (पहिलो संशोिि) अध्यादेश, २०८२ 
 
    राजपत्रमा प्रकाशशि तमति 
  208२।09।०४ 
 
संर्ि् २०82 सालको अध्यादेश िं. 03 
प्रतितिति सभा सदस्य तिर्ाचि ऐि, २०७४ लाई संशोिि गिा बिेको अध्यादेश 
 प्रस्िार्िा  प्रतितिति सभा सदस्य तिर्ाचि ऐि, २०७४ लाई ित्काल संशोिि गिा र्ाञ्छिीय 
भएको र िाल सङ् घीय संसकोको अतिर्ेशि िभएकोले   
 
 
 
िेपालको संहर्िािको िारा ११४ को उपिारा (१) बमोशजम मशरत्रपररषकोको तसफाररसमा 
राष्ट्रपतिबाट यो अध्यादेश जारी भएको छ। 
 
1. 
संशिप् ि िाम र प्रारम्भ  (१) यस अध्यादेशको िाम "प्रतितिति सभा सदस्य तिर्ाचि 
(पहिलो संशोिि) अध्यादेश , २०८२" रिेको छ। 
 
  
(2) यो अध्यादेश िुरुरि प्रारम्भ िुिेछ। 
 
2. 
प्रतितिति सभा सदस्य तिर्ाचि ऐि, २०७४ को अिुसूचीमा संशोिि  प्रतितिति सभा सदस्य 
तिर्ाचि ऐि, २०७४ को अिुसूची-१ को सट्टा देिायको अिुसूची-१ राशिएको छ -

---

अिुसूची-१ 
  (दफा २८ को उपदफा (५) र दफा ६० को उपदफा (६) सँग सम्बशरिि) 
उम्मेदर्ारको बरदसूचीको लातग समार्ेशी आिार 
क्र.सं. 
प्रतितितित्र् िुिे समूि 
प्रतिशि 
१. 
दतलि 
13.44 
२. 
आददर्ासी जिजाति 
28.72 
३. 
िस आया 
30.28 
४. 
मिेशी 
16.15 
५. 
थारु 
6.52 
६. 
मुशस्लम 
4.89

---

**📌 Sources:**

- 📄 `2082.9.2 प्रतिनिधि सभा सदस्य निर्वाचन (पहिलो संशोधन) अध्यादेश,२०८२` — Page 1

- 📄 `2082.9.2 प्रतिनिधि सभा सदस्य निर्वाचन (पहिलो संशोधन) अध्यादेश,२०८२` — Page 2

---

### Usecase 4 — Human Rights Award Fund (Nepali 🇳🇵)

**Scenario:** Human rights researcher understanding fund operations.

**Question:** *मानव अधिकार पुरस्कार कोषको उद्देश्य र सञ्चालन प्रक्रिया के हो?*

*(What is the purpose and process of the Human Rights Award Fund?)*

*(This demo scopes retrieval to the **Human Rights Award Fund rules** PDF only so the answer is not conflated with other Nepali gazette texts in the seed set.)*



In [7]:
import os
import shutil

# Other Nepali PDFs in the seed set can out-rank this rules booklet in hybrid search.
# Build a one-document mini-corpus so this cell demonstrates the correct source.
_hr_pdf = next(
    f
    for f in os.listdir(corpus_dir)
    if f.endswith(".pdf") and "मानव अधिकार" in f
)
_u4_dir = os.path.join(corpus_dir, ".demo_u4_corpus")
shutil.rmtree(_u4_dir, ignore_errors=True)
os.makedirs(_u4_dir, exist_ok=True)
shutil.copy2(os.path.join(corpus_dir, _hr_pdf), os.path.join(_u4_dir, _hr_pdf))

_rag_u4 = GovRAG(
    corpus_dir=_u4_dir,
    config=GovRAGConfig(cache_dir=os.path.join(_u4_dir, ".nepal_gov_cache")),
)

_ = ask_and_display(
    "मानव अधिकार पुरस्कार कोषको उद्देश्य र सञ्चालन प्रक्रिया के हो?",
    context="Human rights researcher understanding fund operations.",
    rag_client=_rag_u4,
    use_llm=True,
)


> 💼 **Scenario:** Human rights researcher understanding fund operations.

**❓ Question:** `मानव अधिकार पुरस्कार कोषको उद्देश्य र सञ्चालन प्रक्रिया के हो?`

---

**📋 Answer:**

मानव अधिकार पुरस्कार कोष सञ्‍चालन धनयमावली, २०७५ 
 
नेपाल‍राजपत्रमा‍प्रकाशित‍धमधत 
2075।09।09 
संिोिन 
1. 
मानव अधिकार पुरस्कार कोष सञ्‍चालन (पहिलो‍संिोिन)‍ 
धनयमावली, २०82 
 
 
 
 
‍‍‍‍‍ ‍‍‍‍2082।09।24 
 
प्रिासकीय  कायहवधि (धनयधमत गने) ऐन, २०१३ को दफा २ ले ददएको अधिकार प्रयोग गरी नेपाल 
सरकारले देिायका धनयमिरु बनाएको छ। 
1. 
संशिप्‍त नाम र प्रारम्भः ‍(१) यी धनयमिरुको नाम "मानव अधिकार पुरस्कार कोष सञ्‍चालन 
धनयमावली, २०७५" रिेको  छ।  
(२) यो धनयमावली तुरुन्त प्रारम्भ िुनेछ।  
२.  
पररभाषाः हवषय वा प्रसङ्गले अको अर्य नलागेमा यस धनयमावलीमा,-  
(क)  "अध्यि" भन्‍नाले सधमधतको अध्यि सम्झनु पछय। 
(ख) 
"उपसधमधत" भन्‍नाले धनयम ८ बमोशजम गदित पुरस्कार धसफाररस उपसधमधत 
सम्झनु पछय। 
(ग) 
"कोष" भन्‍नाले धनयम ३ बमोशजमको मानव अधिकार पुरस्कार अिय कोष 
सम्झनु पछय। 
(घ)  
"पुरस्कार" भन्‍नाले यस धनयमावली बमोशजम हवतरण गररने मानव अधिकार 
पुरस्कार सम्झनु पछय।  
(ङ) 
"मन्त्रालय" भन्‍नाले नेपाल सरकारको कानून न्याय तर्ा संसदीय माधमला 
मन्त्रालय सम्झनु पछय। 
(च)  
"सधमधत" भन्‍नाले धनयम ५ बमोशजमको मानव अधिकार पुरस्कार अिय कोष 
व्यवस्र्ापन सधमधत सम्झनु पछय।  
३.  
कोषको स्र्ापना र सञ्‍चालनः (१) मानव अधिकारको संरिण र संबर्द्यनमा उल्लेखनीय 
योगदान पुर्‍याउने व्यशि वा संस्र्ालाई प्रत्येक दुई‍वषयमा‍एक‍पटक अन्तरायश्‍िय मानव 
अधिकार ददवसको उपलक्ष्यमा नेपाल सरकारको तफयबाट मन्त्रालयले मानव अधिकार 
                                               
  
पहिलो‍संिोिनद्वारा‍संिोधित।

---

पुरस्कार प्रदान गनय मानव अधिकार पुरस्कार अिय कोष नामको एक कोष स्र्ापना 
गररएकोछ।  
(२)  
उपधनयम (१) बमोशजमको कोषमा देिाय बमोशजमका रकमिरु रिने छन्:-  
(क)  नेपाल सरकारबाट प्राप्‍त रकम,  
(ख) 
स्वदेिी व्यशि वा अन्य सङ्‍घ संस्र्ाबाट प्राप्‍त रकम,  
(ग) 
हवदेिी व्यशि, सरकार वा अन्तरायश्‍िय सङ्‍घ वा संस्र्ाबाट प्राप्‍त 
रकम,  
(घ)  
कोषको रकमबाट बढे बढाएको रकम,  
(ङ)  
अन्य स्रोतबाट प्राप्‍त रकम।  
(३)  
उपधनयम (२) को खण्ड (ग) बमोशजम हवदेिी व्यशि, सरकार वा 
अन्तरायश्‍िय सङ्‍घ वा संस्र्ाबाट कुनै रकम प्राप्‍त गनुय अशघ नेपाल सरकार, अर्य मन्त्रालयको 
स्वीकृती धलनु पनेछ।  
(४)  
उपधनयम (१) बमोशजम कोषमा प्राप्‍त िुने रकम नेपाल रा्‍ि बैङ्‍कबाट "क" 
श्रेणीको मान्यता प्राप्‍त वाशणज्य बैङ्‍क मध्येबाट सधमधतले तोकेको बैङ्‍कमा मुद्दती खाता 
खोली जम्मा गररनेछ।  
४.  
कोषको प्रयोग : (१) कोषमा जम्मा भएको रकमबाट आशजयत ब्याज यस धनयमावली बमोशजम 
प्रदान गररने पुरस्कारको प्रयोजनको लाधग प्रयोग गररनेछ।  
(२) कुनै कारणबाट उपधनयम (१) बमोशजमको कायमा कोषको रकम खचय िुन 
नसकेमा त्यस्तो रकम प्रत्येक वषयको पुष मसान्तधभत्र कोषको मूल साँवामा जोड जम्मा गरी 
राशखनेछ।  
(३) मन्त्रालयले कोषको रकमबाट आशजयत ब्याज रकम यस धनयमावली बमोशजम 
प्रदान गररने पुरस्कार बािेक अन्य कायमा प्रयोग गने छैन।  
५.  
सधमधतको गिनः (१) कोषको सञ्‍चालन तर्ा व्यवस्र्ापनको लाधग मन्त्रालयमा देिाय 
बमोशजमको एक कोष व्यवस्र्ापन सधमधत रिनेछः-  
(क)  सशचव, मन्त्रालय  
 
 
 
-‍अध्यि  
(ख)  
सिसशचव (कानून), प्रिानमन्त्री तर्ा  
 
 
मशन्त्रपररषद्को कायायलय  
 
 
-‍सदस्य  
(ग)  
सिसशचव (मानव अधिकार सम्बन्िी हवषय िेने),

---

मन्त्रालय  
 
 
 
 
-‍सदस्य  
(घ)  
सिसशचव, महिला, बालबाधलका तर्ा  
 
 
जे्‍ि नागररक मन्त्रालय  
 
 
-‍सदस्य  
(ङ)  
उपसशचव (मानव अधिकार सम्बन्िी 
 
 
 हवषय िेने), मन्त्रालय  
 
 
- सदस्य सशचव  
 (२)  सधमधतको सशचवालयको काम मन्त्रालयको मानव अधिकार सम्बन्िी काम 
िेने िाखाले गनेछ।  
(३)  
सधमधतको सशचवालयको सञ्‍चालन, पुरस्कार धसफाररस तर्ा हवतरण समारोि 
सम्बन्िी खचय मन्त्रालयको वाहषयक बजेटबाट व्यिोररनेछ।  
६.  
सधमधतको बैिकः‍(१) सधमधतको बैिक वषयको कम्तीमा दुई पटक अध्यिले तोकेको धमधत, 
समय र स्र्ानमा बस्नेछ।  
(२)  सधमधतको बैिकको अध्यिता अध्यिले गनेछ।  
(३)  सधमधतको सदस्य सशचवले सधमधतको बैिक बस्नुभन्दा कम्तीमा चौबीस घण्टा 
अगावै बैिकमा छलफल िुने हवषयसूची सहितको सूचना सबै सदस्यिरुलाई ददनु पनेछ।  
(४)  
अध्यि सहित सधमधतका तीनजना सदस्य उपशस्र्त भएमा बैिकको लाधग 
गणपूरक संख्या पुगेको माधननेछ।  
(५)  
सधमधतको बैिकमा पुरस्कार प्रदान सम्बन्िी धनणयय गनुय पदाय सवयसम्मत 
रुपमा गनुय पनेछ।  
(६)  
बैिकमा आवश्यकता अनुसार मानव अधिकार सम्बन्िी हवज्ञ वा आवश्यक 
अन्य पदाधिकारीलाई समेत आमन्त्रण गनय सहकनेछ।  
(७)  
सधमधतको सदस्य सशचवले बैिकको धनणयय अधभलेख गरी अध्यिबाट 
प्रमाशणत गराई राख्‍नु पनेछ।  
(८)  
सधमधतको बैिक सम्बन्िी अन्य कायहवधि सधमधत आफैले धनिायरण गरे 
बमोशजम िुनेछ।  
७.  
सधमधतको काम, कतयव्य र अधिकारः सधमधतको काम, कतयव्य र अधिकार देिाय बमोशजम 
िुनेछ:-  
(क)  कोषको सञ्‍चालन तर्ा व्यवस्र्ापन सम्बन्िमा आवश्यक नीधत धनिायरण गने.

---

(ख)  
कोषको रकम व्यवशस्र्त गने.  
(ग) 
पुरस्कारको स्वरुप तर्ा त्यसको रािी तोक्ने,  
(घ)  
पुरस्कार प्रदान गररने िेत्र तर्ा त्यसको आिार धनिायरण गने,  
(ङ)  
उपसधमधतले धसफाररस गरेका व्यशि वा संस्र्ािरु मध्येबाट उपयुि 
देशखएका व्यशि वा संस्र्ालाई पुरस्कार प्रदान गनय मन्त्रालय समि 
धसफाररस गने,  
(च) 
पुरस्कार प्रदान गदाय ददइने प्रमाणपत्रको ढाँचा स्वीकृत गने,  
(छ)  सधमधतले गनुय पने काम कारबािी सुचारु रुपले सञ्‍चालन गनय आवश्यकता 
अनुसार उपसधमधत गिन गने,  
(ज)  
कोष सञ्‍चालन तर्ा व्यवस्र्ापन सम्बन्िी अन्य आवश्यक कायिरू गने।  
८.  
उपसधमधतको गिनः  (१) यस धनयमावली बमोशजम पुरस्कार पाउने व्यशि वा संस्र्ाको नाम 
छनौट गरी सधमधत समि धसफाररस गनय सधमधतले अध्यि वा सधमधतको कुनै सदस्यको 
संयोजकत्वमा देिायका िेत्रिरुसाँग सम्बशन्ित हवज्ञिरू समेत रिेको बढीमा पाँच सदस्यीय 
उपसधमधत गिन गनय सक्नेछ:-  
(क)  मानव अधिकारको हवषयमा अध्ययन, अध्यापन तर्ा अनुसन्िान,  
(ख)  
कानून व्यवसाय,  
(ग)  
बालबाधलका, महिला, जे्‍ि नागररक, दधलत, अपाङ्गता भएका 
व्यशि, हपछधडएका वा जोशखममा रिेका वा धसमान्तीकृत समूिको 
मानव अधिकार संरिणको िेत्रमा गरेको योगदान,  
(घ)  
लैहङ्गक न्याय,  
(ङ)  
आम सञ्‍चार।  
(२)  उपसधमधतको काम, कतयव्य र अधिकार तर्ा काय अवधि उपसधमधत गिन 
गदायका बखत सधमधतले तोके बमोशजम िुनेछ।  
(३)  उपसधमधतको बैिक उपसधमधतको संयोजकले तोकेको धमधत, समय र 
स्र्ानमा आवश्यकता अनुसार बस्नेछ।  
(४)  उपसधमधतको बैिक सम्बन्िी अन्य कायहवधि उपसधमधत आफैले धनिायरण गरे 
बमोशजम िुनेछ।  
९.  
वस्तुधन्‍ि आिारमा नाम धसफाररस गनुय पनेः  (१) उपसधमधतले कुनै व्यशि वा संस्र्ालाई 
पुरस्कारको लाधग धसफाररस गदाय त्यस्तो व्यशि वा संस्र्ाले मानव अधिकारको संरिण र

---

संबर्द्यनको लाधग पुर्‍याएको िोस योगदान, त्यस्तो व्यशि वा संस्र्ाको काय िेत्र, हविेषज्ञता, 
नैधतकता, मानव अधिकार र मानवता प्रधतको समपयण जस्ता कुरािरु स्प्‍ट रुपमा खुलाई 
वस्तुधन्‍ि आिारमा नाम धसफाररस गनुय पनेछ।  
(२) उपधनयम (१) बमोशजम उपसधमधतले पुरस्कारको धसफाररस गदाय मानव 
अधिकारसाँग सम्बशन्ित िेत्रमध्ये कुन िेत्रमा उल्लेखनीय योगदान पुर्‍याएको िो सो कुरा 
समेत खुलाई बढीमा तीनजना व्यशि वा संस्र्ाको नाम धसफाररस गनुय पनेछ।  
१०.  
पुरस्कार धसफाररसका आिार: (१) यस धनयमावली बमोशजम कुनै व्यशि वा संस्र्ालाई 
पुरस्कारको लाधग धसफाररस गदाय त्यस्तो व्यशि वा संस्र्ाले मानव अधिकारको संरिण र 
संबर्द्यनमा पुर्‍याएको योगदानको आिारमा पुरस्कारको धसफाररस गनुय पनेछ।  
(२)  उपधनयम (१) बमोशजम पुरस्कृत िुने व्यशि वा संस्र्ाको योगदानको 
मूल्याङ्कन गदाय हवचार गनुय पने कुरािरु र त्यसको आिार सधमधतले धनिायरण गरे बमोशजम 
िुनेछ।  
 
(३)  पुरस्कारको लाधग व्यशिको नाम धसफाररस गदाय उमेर, िमय, वणय, धलङ्ग, जात, 
जाधत, लैहङ्गकता, राजनीधतक वा वैचाररक आस्र्ा जस्ता कुरािरुको आिारमा भेदभाव गनुय 
िुाँदैन।  
(४)  सधमधतले पुरस्कारको लाधग व्यशि वा संस्र्ाको नाम धसफाररस गनुय अशघ 
पुरस्कारको लाधग योग्य व्यशि वा संस्र्ाको नाम सूचनामा तोहकएको समयावधिधभत्र उपलब्ि 
गराउन कम्तीमा दुईवटा राश्‍ियस्तरका दैधनक पधत्रकामा सूचना प्रकािन गरी सावयजधनक 
रुपमा आव्िान गनुय पनेछ र सो सूचना मन्त्रालयको वेबसाइटमा समेत प्रकािन गनुय पनेछ।  
११.  
पुरस्कारको लाधग धसफाररस गनय निुने :‍ (१) यस धनयमावलीमा अन्यत्र जुनसुकै कुरा 
लेशखएको भए तापधन मानव अधिकार उल्लङ्घन सम्बन्िी फौजदारी कसूरमा सजाय पाएको, 
प्रचधलत कानून बमोशजम दामासािी वा कालो सूचीमा परेको, बाल हिंसा, घरेलु वा लैहङ्गक हिंसा 
गरेको कारणले सजाय पाएको वा िधतपूधतय धतरेको, प्रचधलत कानून बमोशजम कर चुिा 
नगरेको कारणबाट कारबािीमा परेको व्यशि वा संस्र्ालाई पुरस्कारको लाधग धसफाररस गनय 
िुाँदैन।  
(२)  उपधनयम (१) बमोशजम पुरस्कारको लाधग धसफाररस गनय निुने व्यशि वा 
संस्र्ालाई पुरस्कारको धसफाररस भई धनजले त्यस्तो पुरस्कार प्राप्‍त गरेमा मन्त्रालयले जुनसुकै 
बखत त्यस्तो पुरस्कार रकम र सो सम्बन्िी प्रमाणपत्र हफताय गनय लगाउन वा सो बराबरको

---

**📌 Sources:**

- 📄 `मानव अधिकार पुरस्कार कोष सञ्चालन नियमावली, 2075` — Page 1

- 📄 `मानव अधिकार पुरस्कार कोष सञ्चालन नियमावली, 2075` — Page 2

- 📄 `मानव अधिकार पुरस्कार कोष सञ्चालन नियमावली, 2075` — Page 3

---

## ✏️ Try Your Own Question

Type any question below in English or Nepali and run the cell.



In [8]:
# Change this to anything you want to ask
YOUR_QUESTION = "What role does the National AI Centre play in Nepal?"

# Examples in Nepali:
# YOUR_QUESTION = "नेपालको संविधानमा शिक्षाको अधिकार कसरी परिभाषित गरिएको छ?"
# YOUR_QUESTION = "डिजिटल नेपाल फ्रेमवर्कको उद्देश्य के हो?"

_ = ask_and_display(YOUR_QUESTION)



**❓ Question:** `What role does the National AI Centre play in Nepal?`

---

**📋 Answer:**

Provided that nothing shall be deemed to prevent the regulation, by making law, 
of the operation and protection of religious sites and religious trusts and management 
of trust properties and land. 
(3) No person shall, in the exercise of the right conferred by this Article, do, or 
cause to be done, any act which may be contrary to public health, decency and 
morality or breach of public peace, or convert another person from one religion to 
another or any act or conduct that may jeopardize ot…

*[See source documents for full text]*

---

**📌 Sources:**

- 📄 `Constitution of Nepal (2nd amd. English)` — Page 15

- 📄 `Constitution of Nepal (2nd amd. English)` — Page 16

- 📄 `Constitution of Nepal (2nd amd. English)` — Page 172

---

## 📊 Retrieval Validation

These are real benchmark results run against the 5-document seed corpus.
Recall@3 = 0.857 means: for 6 out of 7 test queries, the correct document
section appeared in the top 3 retrieved results.

> This measures **retrieval quality only** — not answer safety or correctness.
> See [Scope and Limitations](https://github.com/irfanalidv/Nepal-Gov-Agent#scope-and-limitations).



In [9]:
import logging

logging.getLogger("nepal_gov_agent").setLevel(logging.WARNING)

from nepal_gov_agent import run_benchmark

results = run_benchmark(rag, verbose=False)
print(results.report())



Nepal GovAgent Benchmark Results
Total queries:      7
Recall@1:           0.714
Recall@3:           1.000
Recall@5:           1.000
Keyword hit rate:   1.000
Doc hit rate:       1.000
Nepali recall@3:    1.000
English recall@3:   1.000


---

## 📂 Use Your Own Nepal Government PDFs

This library works with **any** Nepal government PDF — not just the 5 seed
documents. Ministry circulars, municipal SOPs, land records, provincial
guidelines — drop them in and ask questions.



In [10]:
import os

# Option 1 — Upload from your computer (Google Colab only)
try:
    from google.colab import files

    print("Select your Nepal government PDF(s) to upload...")
    uploaded = files.upload()

    os.makedirs("./my_ministry_docs/", exist_ok=True)
    for fname in uploaded:
        dest = os.path.join("./my_ministry_docs/", fname)
        with open(dest, "wb") as f:
            f.write(uploaded[fname])
        print(f"✅ Added: {fname}")

    print("\n🔄 Now re-run the Setup cell pointing at your folder:")
    print('   rag = GovRAG(corpus_dir="./my_ministry_docs/")')
except ImportError:
    print("Not in Google Colab — copy PDFs into a folder locally, then:")
    print('   rag = GovRAG(corpus_dir="./my_ministry_docs/")')

# Option 2 — Point at a folder you already have
# rag = GovRAG(corpus_dir="./my_ministry_docs/")



Not in Google Colab — copy PDFs into a folder locally, then:
   rag = GovRAG(corpus_dir="./my_ministry_docs/")


---

## 🤝 Contribute

The more documents in the corpus, the more useful this becomes.

**What's needed most:**
- Ministry circulars (2080–2082 BS)
- Provincial government SOPs
- Municipality service guidelines
- Land, labor, tax regulation PDFs

Open a PR: **[github.com/irfanalidv/Nepal-Gov-Agent](https://github.com/irfanalidv/Nepal-Gov-Agent)**

---

<div align="center">

**Built in the spirit of AIAN's four pillars: Data. Infrastructure. Policy. Resources.**

🇳🇵 *Working on Nepal's AI infrastructure layer? Open an issue or reach out.*

**Irfan Ali** · [LinkedIn](https://www.linkedin.com/in/irfanalidv/) ·
[GitHub](https://github.com/irfanalidv) · irfan.ali@datacortex.in

</div>

